In [ ]:
from openai import OpenAI
from pydantic import BaseModel
from dataclasses import dataclass
import os
import json
from typing import Literal

@dataclass(frozen=True)

class Provider:

    name:str
    is_free:bool
    env_var:str
    base_url:str |None
    model:str

PROVIDERS=[
    Provider(
        name="OpenAI",
        is_free=False,
        env_var="OPENAI_API_KEY",
        base_url=None,
        model="gpt-4o-mini"
    ),

    Provider(
        
        name="Groq",
        is_free=True,
        env_var="GROQ_API_KEY",
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-120b"
    ),

    Provider(
        
        name="Open Router",
        is_free=False,
        env_var="OPENROUTER_API_KEY",
        base_url="https://openrouter.ai/api/v1",
        model="nvidia/llama-nemotron-rerank-vl-1b-v2:free"
    )
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)    
    raise ValueError(f"No valid provider found. Please set one of the following environment variables: {expected}")

def build_client(provider:Provider) ->OpenAI:
    api_key=os.environ.get(provider.env_var)

    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(api_key=api_key, base_url=provider.base_url)

def have_any_key() -> bool:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return True
print("Found a provider key." if have_any_key() else "No provider key found.")


Found a provider key.


In [7]:
SYSTEM_PROMPT= """
You are a product-review sentiment analyst.

Read each review the user sends and classify its sentiment. We should also attach a confidence score for the sentiment
between 0 to 1 and a short explanation for the sentiment as a reason.

For each review, respond with ONLY a JSON object -  no other text or comments - in exactly the following shape:

{
    "sentiment": "positive" | "negative" | "neutral",
    "confidence": 0.0 to 1.0,
    "reason": "short explanation for the sentiment"
}

""".strip()

In [8]:
def analyze_review(incoming_review:str)-> str:
    provider=select_provider()
    client=build_client(provider)


    result=client.chat.completions.create(
        model=provider.model,
        max_tokens=100,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": incoming_review},
        ],
    )

    return result.choices[0].message.content.strip()



In [9]:
REVIEW=[
        "Absolutely love these headphones — great sound and the battery lasts all day!",
        "Arrived broken and customer service never replied. Very disappointed.",
        "It works. Nothing special, nothing terrible.",
    ]

In [10]:
analyze_review(REVIEW[1])

'{\n    "sentiment": "negative",\n    "confidence": 0.98,\n    "reason": "The review mentions receiving a broken product and lack of customer service response, indicating strong dissatisfaction."\n}'

In [11]:
review1=analyze_review(REVIEW[1])

In [12]:
class SentimentResult(BaseModel):                                  #it builds the json to object
    sentiment:Literal["positive" ,"negative" , "neutral"] 
    confidence:float
    reason:str                                     #accepted input is string
    


NameError: name 'Literal' is not defined

In [ ]:
def validate_sentiment_reply(raw_reply :str)-> SentimentResult | str:
    try:
        parsed=json.loads(raw_reply)
        return SentimentResult(**parsed)

    except json.JSONDecodeError:
        return f"Invalid JSON: {raw_reply}"

In [14]:
result= validate_sentiment_reply(review1)

NameError: name 'validate_sentiment_reply' is not defined

In [13]:
print(type(result))

NameError: name 'result' is not defined

In [ ]:
def route_by_sentiment(result: SentimentResult) -> str:
    if result.sentiment == "positive":
        return "Positive review"
    elif result.sentiment == "negative":
        return "Negative review, connect with customer service"
    else:
        return "Neutral review, no action needed"


route_by_sentiment(result)
    